# Orchestrator Agent

Coordinates the planner and travel agents via a workflow graph.
Uses MCP to discover agents and routes tasks through them in sequence.

In [1]:
import json
import os
from collections.abc import AsyncIterable

import anthropic
import uvicorn
import weave
from a2a.types import TaskState
from dotenv import load_dotenv
from google.protobuf import json_format

import prompts
from common import BaseAgent, build_a2a_app
from workflow import Status, WorkflowGraph, WorkflowNode

load_dotenv()

WANDB_PROJECT = os.getenv('WANDB_WORKSPACE')
weave_client = weave.init(WANDB_PROJECT)
print(f'Weave initialized: {WANDB_PROJECT}')

/Users/paul/.cache/uv/archive-v0/pqxv7ELpHFQOMM0CmHnVk/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
weave: weave version 0.52.37 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: paulbruffett.
weave: View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave


Weave initialized: pbruffett/a2a-lab


In [2]:
class OrchestratorAgent(BaseAgent):
    def __init__(self):
        super().__init__(agent_name='Orchestrator Agent', description='Facilitate inter agent communication', content_types=['text', 'text/plain'])
        self.graph = None
        self.results = []
        self.travel_context = {}
        self.query_history = []
        self.context_id = None

    @weave.op()
    async def _generate_summary(self) -> str:
        client = anthropic.AsyncAnthropic()
        response = await client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=16000,
            thinking={'type': 'adaptive'},
            messages=[{
                'role': 'user',
                'content': prompts.SUMMARY_COT_INSTRUCTIONS.replace('{travel_data}', str(self.results)),
            }],
        )
        return next((b.text for b in response.content if b.type == 'text'), '')

    @weave.op()
    def _answer_user_question(self, question) -> str:
        try:
            client = anthropic.Anthropic()
            response = client.messages.create(
                model='claude-sonnet-4-6',
                max_tokens=16000,
                messages=[{
                    'role': 'user',
                    'content': prompts.QA_COT_PROMPT
                        .replace('{TRIP_CONTEXT}', str(self.travel_context))
                        .replace('{CONVERSATION_HISTORY}', str(self.query_history))
                        .replace('{TRIP_QUESTION}', question),
                }],
                output_config={
                    'format': {
                        'type': 'json_schema',
                        'schema': {
                            'type': 'object',
                            'properties': {
                                'can_answer': {'type': 'string', 'enum': ['yes', 'no']},
                                'answer': {'type': 'string'},
                            },
                            'required': ['can_answer', 'answer'],
                            'additionalProperties': False,
                        },
                    },
                },
            )
            return next((b.text for b in response.content if b.type == 'text'), '')
        except Exception:
            return '{"can_answer": "no", "answer": "Cannot answer based on provided context"}'

    def _update_node(self, node_id, **attrs):
        self.graph.set_node_attributes(node_id, {k: v for k, v in attrs.items() if v is not None})

    def _add_graph_node(self, task_id, context_id, query, parent_id=None, node_key=None):
        node = WorkflowNode(task=query, node_key=node_key)
        self.graph.add_node(node)
        if parent_id:
            self.graph.add_edge(parent_id, node.id)
        self._update_node(node.id, task_id=task_id, context_id=context_id, query=query)
        return node

    def _clear_state(self):
        self.graph = None
        self.results.clear()
        self.travel_context.clear()
        self.query_history.clear()

    @weave.op()
    async def stream(self, query, context_id, task_id) -> AsyncIterable[dict[str, any]]:
        if not query:
            raise ValueError('Query cannot be empty')
        if self.context_id != context_id:
            self._clear_state()
            self.context_id = context_id

        self.query_history.append(query)

        # Determine starting node
        if not self.graph:
            self.graph = WorkflowGraph()
            start_node = self._add_graph_node(task_id, context_id, query, node_key='planner')
            print(f'[Orchestrator] New workflow started for: "{query[:80]}"')
        elif self.graph.state == Status.PAUSED:
            start_node_id = self.graph.paused_node_id
            self._update_node(start_node_id, query=query)
            start_node = self.graph.nodes[start_node_id]
            print(f'[Orchestrator] Resuming paused workflow')
        else:
            return

        current_node_id = start_node.id

        while True:
            self._update_node(current_node_id, task_id=task_id, context_id=context_id)
            resume_node_id = None

            async for stream_resp, task in self.graph.run_workflow(current_node_id):
                payload_type = stream_resp.WhichOneof('payload')

                if payload_type == 'status_update':
                    evt = stream_resp.status_update
                    context_id = evt.context_id
                    if evt.status.state == TaskState.TASK_STATE_COMPLETED and context_id:
                        continue
                    if evt.status.state == TaskState.TASK_STATE_INPUT_REQUIRED:
                        question = evt.status.message.parts[0].text
                        print(f'[Orchestrator] Agent asked: "{question[:80]}"')
                        try:
                            answer = json.loads(self._answer_user_question(question))
                            if answer['can_answer'] == 'yes':
                                print(f'[Orchestrator] Auto-answered from context: "{answer["answer"][:60]}"')
                                resume_node_id = self.graph.paused_node_id
                                self._update_node(resume_node_id, query=answer['answer'])
                                break
                        except Exception:
                            pass

                elif payload_type == 'artifact_update':
                    artifact = stream_resp.artifact_update.artifact
                    self.results.append(artifact)
                    if artifact.name == 'PlannerAgent-result':
                        artifact_data = json_format.MessageToDict(artifact.parts[0].data)
                        if 'trip_info' in artifact_data:
                            self.travel_context = artifact_data['trip_info']
                        parent_id = current_node_id
                        first_task_node = None
                        for task_data in artifact_data['tasks']:
                            node = self._add_graph_node(task_id, context_id, task_data['description'], parent_id=parent_id)
                            print(f'[Orchestrator] Planned task: "{task_data["description"][:70]}"')
                            if not first_task_node:
                                first_task_node = node
                            parent_id = node.id
                        print(self.graph)
                        if first_task_node:
                            resume_node_id = first_task_node.id
                            break
                    else:
                        continue

                if not resume_node_id:
                    yield (stream_resp, task)

            if resume_node_id:
                current_node_id = resume_node_id
            else:
                break

        if self.graph.state == Status.COMPLETED:
            print('[Orchestrator] All tasks complete, generating summary...')
            summary = await self._generate_summary()
            self._clear_state()
            yield {'response_type': 'text', 'is_task_complete': True, 'require_user_input': False, 'content': summary}

In [ ]:

app = build_a2a_app(OrchestratorAgent(), 'agent_cards/orchestrator_agent.json')

if __name__ == "__main__":
    config = uvicorn.Config(app=app, host='localhost', port=10101)
    server = uvicorn.Server(config)
    await server.serve()

INFO:     Started server process [27574]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:10101 (Press CTRL+C to quit)


INFO:     ::1:60393 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     ::1:60394 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     ::1:60394 - "POST / HTTP/1.1" 200 OK


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019dac23-fcc1-77d6-940e-b384e28a09f3


[Orchestrator] New workflow started for: "Plan my trip to London from San Francisco from 2026-05-01 to 05-11 - staying in "
[Workflow] Running 1 node(s)
[Workflow] Graph state: (RUNNING)
  [READY    ] planner: "Plan my trip to London from San Francisc..."
[Workflow] Executing node planner: "Plan my trip to London from San Francisco from 2026-05-01 to..."
[MCP] Looking up planner resource at mcp://localhost:10100 ...
[MCP] Found planner agent: Langraph Planner Agent @ http://localhost:10102/
[A2A] Sending to Langraph Planner Agent: "Plan my trip to London from San Francisco from 2026-05-01 to 05-11 - staying in ..."
[A2A] Langraph Planner Agent status: TASK_STATE_INPUT_REQUIRED
[Orchestrator] Agent asked: "What is your total budget for this trip?"
INFO:     ::1:60404 - "POST / HTTP/1.1" 200 OK


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019dac24-34e5-75d8-af12-b26d0b1cc220


...

[Orchestrator] Resuming paused workflow
[Workflow] Running 1 node(s)
[Workflow] Graph state: (RUNNING)
  [PAUSED   ] planner: "Plan my trip to London from San Francisc..."
[Workflow] Executing node planner: "Plan my trip to London from San Francisco from 2026-05-01 to..."
[MCP] Looking up planner resource at mcp://localhost:10100 ...
[MCP] Found planner agent: Langraph Planner Agent @ http://localhost:10102/
[A2A] Sending to Langraph Planner Agent: "no..."
[A2A] Received artifact from Langraph Planner Agent: PlannerAgent-result
[Orchestrator] Planned task: "Book round-trip premium economy class air tickets from San Francisco ("
[Orchestrator] Planned task: "Book a suite room at a hotel in London for 2 guests, with a check-in d"
[Orchestrator] Planned task: "No car rental required for this trip."
[Workflow] Graph state: (RUNNING)
  [RUNNING  ] planner: "Plan my trip to London from San Francisc..."
       ↓
  [READY    ] f0e1e71c: "Book round-trip premium economy class ai..."
       ↓
  

weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019dac35-4e68-7897-b279-fd42bdbd4903


[Orchestrator] Resuming paused workflow
[Workflow] Running 3 node(s)
[Workflow] Graph state: (RUNNING)
  [RUNNING  ] planner: "Plan my trip to London from San Francisc..."
       ↓
  [PAUSED   ] f0e1e71c: "Book round-trip premium economy class ai..."
       ↓
  [READY    ] 1df9c725: "Book a suite room at a hotel in London f..."
       ↓
  [READY    ] 646957b4: "No car rental required for this trip."
[Workflow] Executing node f0e1e71c: "Book round-trip premium economy class air tickets from San F..."
[MCP] Searching for agent to handle: "Book round-trip premium economy class air tickets from San Francisco (SFO) to Lo..."
[MCP] Matched agent: Air Ticketing Agent @ http://localhost:10103/
[A2A] Sending to Air Ticketing Agent: "economy..."
[A2A] Air Ticketing Agent status: TASK_STATE_WORKING
[A2A] Air Ticketing Agent status: TASK_STATE_WORKING
[A2A] Received artifact from Air Ticketing Agent: AirTicketingAgent-result
[A2A] Air Ticketing Agent status: TASK_STATE_COMPLETED
[Workflow] Graph s

weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019dac49-d61a-7a05-b913-848352d2b45c


[Orchestrator] New workflow started for: "Plan my trip to London from San Francisco from 2026-05-01 to 05-11 - staying in "
[Workflow] Running 1 node(s)
[Workflow] Graph state: (RUNNING)
  [READY    ] planner: "Plan my trip to London from San Francisc..."
[Workflow] Executing node planner: "Plan my trip to London from San Francisco from 2026-05-01 to..."
[MCP] Looking up planner resource at mcp://localhost:10100 ...
[MCP] Found planner agent: Langraph Planner Agent @ http://localhost:10102/
[A2A] Sending to Langraph Planner Agent: "Plan my trip to London from San Francisco from 2026-05-01 to 05-11 - staying in ..."
[A2A] Langraph Planner Agent status: TASK_STATE_INPUT_REQUIRED
[Orchestrator] Agent asked: "Great! I have a good start on your trip plan. What is your total budget for this"
INFO:     ::1:60794 - "POST / HTTP/1.1" 200 OK
